In [20]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import tensor

import numpy as np
import pandas as pd

# First attempt outline

Dataset:
- create custom pytorch dataset object that is then compatible with data_loader

Want to create a neural network that predicts a single genes expression across samples from all TFs
- will not subset to network yet to get hang of pytorch
- will not use MML yet to get hang of pytorch

X: Expression of all TFs in all samples
Y: Expression of single gene in all samples

Activation function: linear activation function



### Creating dataset object

In [21]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [37]:
DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'
TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)

In [38]:
TF_expressions.shape

(15935, 1198)

In [22]:
from torch.utils.data import Dataset


class CustomTFGE(Dataset):
    def __init__(self, device, transform=None, target_transform=None):
        #load the two tsv files
        self.DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'
        self.TF_expressions = pd.read_csv((f"{self.DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)
        self.gene_expressions = pd.read_csv((f"{self.DATA_ROOT}/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)

        #no transforms needed so set to none
        self.transform = transform
        self.target_transform = target_transform

        #convert to torch tensors
        self.TF_expressions = torch.tensor(np.asarray(self.TF_expressions).T, dtype = torch.float32, device = device)
        self.gene_expressions = torch.tensor(np.asarray(self.gene_expressions), dtype = torch.float32, device = device)

    def __len__(self):
        return self.TF_expressions.shape[0]

    def __getitem__(self, idx):
        #in this case want to always retrieve the same gene (target) but a different sample containg all the TF values
        TFs_exp = self.TF_expressions[:, idx]
        Gene_exp = self.gene_expressions[:, 1]
        return TFs_exp, Gene_exp

In [23]:
dataset = CustomTFGE(device)
dataset

In [24]:
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

In [30]:
#defining the neural network

class BasicNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            #linear activation layer takes 1198 TFs per sample -> outputs a value for the TF for all 15935 samples
            nn.Linear(1198, 15935)
        )

    def forward(self, x):
        expressions = self.linear_relu_stack(x)
        return(expressions)

In [33]:
model = BasicNeuralNetwork().to(device)
print(model)

BasicNeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=1198, out_features=15935, bias=True)
  )
)


## Train test loop

In [48]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        
        loss = loss.item()
        print(f"loss: {loss:>7f}")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            print(f"MSE on test dataset is {loss_fn(pred, y).item()}")
            


In [45]:
learning_rate = 1e-3
batch_size = 15935
epochs = 20

# Initialize the loss function
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [46]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [49]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.060426
MSE on test dataset is 0.09641031175851822
Epoch 2
-------------------------------
loss: 0.052984
MSE on test dataset is 0.09244468808174133
Epoch 3
-------------------------------
loss: 0.048744
MSE on test dataset is 0.09346690773963928
Epoch 4
-------------------------------
loss: 0.049659
MSE on test dataset is 0.09564613550901413
Epoch 5
-------------------------------
loss: 0.051800
MSE on test dataset is 0.09525609016418457
Epoch 6
-------------------------------
loss: 0.051275
MSE on test dataset is 0.09247411787509918
Epoch 7
-------------------------------
loss: 0.048247
MSE on test dataset is 0.08986686170101166
Epoch 8
-------------------------------
loss: 0.045373
MSE on test dataset is 0.08902442455291748
Epoch 9
-------------------------------
loss: 0.044306
MSE on test dataset is 0.08945062011480331
Epoch 10
-------------------------------
loss: 0.044548
MSE on test dataset is 0.08969587832689285
Epoch 11
----------